# Make a training consensus from 1st order models, for 2nd order training

We want to further improve the 1st order models, by additional fine-tuning on real data. But we don't have any ground truth transcriptions for the real images.

So we're going to make transcriptions with all 5 of the first order models, and them make a consensus marking where those transcriptions agree between the models, and then fine-tune further on the cases where there is consensus between the models.

After completing this notebook, we will have a consensus-ground-truth training dataset: 1000 images, transcribed values for each image, and a flag for each transcribed value

In [2]:
# Step 1. Get a sample of real images 
#
# We have 634,893 images - pick about 1000 of them and 
#   set them up as a training dataset:

import subprocess


# Location of main image dataset, already sized and filtered
SOURCE = "/data/scratch/philip.brohan/documents/Daily_Rainfall_UK/jpgs_25pc_filtered"
# Location to save sampled images and transcriptions
OUTPUT = "../../consensus_data/1st_order"
# Output directory on Azure ML datastore
AML_OUTPUT = "consensus_data/1st_order"
# Number of images to sample
COUNT = 1000


In [ ]:

subprocess.run(
    [
        "python",
        "../../scripts/sample_unseen_images.py",
        "--source-root",
        SOURCE,
        "--output-root", 
        OUTPUT,
        "--count",
        str(COUNT),
        "--seed", # Random seed for reproducibility
        "42",
    ],
    check=True,
)


In [13]:
# Upload the sampled images to the Azure ML datastore:
# Deleting the existing contents of the datastore directory first, if necessary
subprocess.run(
    [
        "bash",
        "../../scripts/aml_delete.sh",
        AML_OUTPUT,
    ],
    check=True,
)
subprocess.run(
    [
        "bash",
        "../../scripts/aml_upload.sh",
        "--src",
        OUTPUT,
        "--dst",
        AML_OUTPUT,
    ],
    check=True,
)   


Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default
Deleting contents under datastore path: consensus_data/1st_order/
Done. Deleted contents under consensus_data/1st_order/
Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

Uploading  ../../consensus_data/1st_order
        → https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order


Finished[#############################################################]  100.0000%.jpg"[]  100.0000%.0000%0%jpg"[]  100.0000%%000%"[]  100.0000%


[
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/sample_manifest.csv",
    "Last Modified": "2026-06-26T13:11:39+00:00",
    "Type": "text/csv",
    "eTag": "\"0x8DED38475B10AC6\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/sample_manifest.jsonl",
    "Last Modified": "2026-06-26T13:11:39+00:00",
    "Type": null,
    "eTag": "\"0x8DED38475C6645E\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/images/DRain_1881-1890_Cumberland-60.jpg",
    "Last Modified": "2026-06-26T13:11:39+00:00",
    "Type": "image/jpeg",
    "eTag": "\"0x8DED38475D751DE\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/images/DRain_1891-1900_RainNos_Surrey_Box_2_M-W_B001-0.jpg",
    "Last Modified": "2026-06-26T13:11:40+00:00",
    "Type": "image/jpeg",
    "eTag": "\"0x8DED38475E35E0D\""
  }

CompletedProcess(args=['bash', '../../scripts/aml_upload.sh', '--src', '../../consensus_data/1st_order', '--dst', 'consensus_data/1st_order'], returncode=0)

That's got the images selected and uploaded. Next step is to run an extraction on those images with each of the 1st order models

In [ ]:
# Basic setup - model names, batch sizes, etc.

import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, model_slug, batch_size, total_shards)
MODEL_SETTINGS = [
    ("SmolVLM", "smolvlm2", 50, 1),
    ("Granite4", "granite4", 50, 1),
    ("Gemma3", "gemma3", 50, 1),
    ("Gemma4", "gemma4", 50, 1),
    ("Ministral", "ministral", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

# Images used for training the checkpoint models. Fake data for the 1st order models.
# Must match the dataset used in finetune_original_models.ipynb
DATASET_DIR = "fake_daily_rainfall_2"
TRAINING_IMAGES_PATH = f"{DATASET_DIR.rstrip('/')}/images"

print(f"Checkpoint training dataset path: {TRAINING_IMAGES_PATH}")
print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Checkpoint training dataset path: fake_daily_rainfall_2/images
Checkpoint training transcriptions path: fake_daily_rainfall_2/transcriptions
Node GPU workers per extraction job: 8


In [ ]:
# Now find the checkpoint for the 1st order version of each model

# Canonical registry location (shared across all notebooks and scripts)
REGISTRY_PATH = Path("../../outputs/model_registry.json")
if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry not found: {REGISTRY_PATH.resolve()}")


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")
def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")



def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _fix_checkpoint_path(path: str) -> str:
    """Keep canonical checkpoint path from the registry.
    
    aml_submit.sh resolves this path relative to AML_DATASTORE_BASE.
    Stripping Daily_rainfall_sample/ can break checkpoint mounts when
    the datastore base is rooted above that folder.
    """
    return _norm_rel(path)


registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
# Support both registry schemas (though now we only have one):
# 1) {"checkpoints": [...]} with keys model_slug/training_dataset_path
# 2) {"models": [...]} with keys base_model/dataset
entries = registry.get("checkpoints")
if not isinstance(entries, list):
    entries = registry.get("models", [])

MODEL_JOBS = []
missing = []
expected_dataset_norm = _norm_rel(TRAINING_IMAGES_PATH)

for label, model_slug, batch_size, total_shards in MODEL_SETTINGS:
    candidates = []
    for e in entries:
        entry_model = str(e.get("model_slug", e.get("base_model", "")))
        entry_dataset = str(
            e.get("training_dataset_path", e.get("dataset", ""))
        )
        checkpoint_path = e.get("checkpoint_path")

        if (
            entry_model == model_slug
            and _norm_rel(entry_dataset) == expected_dataset_norm
            and checkpoint_path
        ):
            candidates.append(e)

    if not candidates:
        missing.append(label)
        continue

    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )

    raw_checkpoint_path = str(best["checkpoint_path"])
    checkpoint_path = _fix_checkpoint_path(raw_checkpoint_path)
    MODEL_JOBS.append((label, checkpoint_path, batch_size, total_shards))
    print(f"{label}: {checkpoint_path}")

if missing:
    raise RuntimeError(
        "Missing fine-tuned checkpoints for: "
        + ", ".join(missing)
        + ". Check DATASET_DIR / DATASET_NAME and ensure model_registry has entries for this dataset."
    )

print(f"\nUsing canonical registry: {REGISTRY_PATH.resolve()}")
print(f"Discovered {len(MODEL_JOBS)} checkpoints")

SmolVLM: Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260625-094936/HuggingFaceTB--SmolVLM2-2.2B-Instruct
Granite4: Daily_rainfall_sample/outputs/checkpoints/granite4-20260625-095008/ibm-granite--granite-vision-4.1-4b
Gemma3: Daily_rainfall_sample/outputs/checkpoints/gemma3-20260625-095027/google--gemma-3-4b-it
Gemma4: Daily_rainfall_sample/outputs/checkpoints/gemma4-20260625-095044/google--gemma-4-E4B-it
Ministral: Daily_rainfall_sample/outputs/checkpoints/ministral-20260625-095102/mistralai--Mistral-Small-3.1-24B-Instruct-2503

Using canonical registry: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/model_registry.json
Discovered 5 checkpoints


In [17]:
# Now run an extraction job, with each checkpoint, on the image sample.

for model_name, checkpoint, batch_size, total_shards in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            AML_OUTPUT + "/images",
            "--transcriptions-path",
            AML_OUTPUT + "/transcriptions",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/consensus_data/1st_order/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260625-094936/H

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

maroon_nerve_wc13kbml4c
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260626-132115
  Model: smolvlm
  Dataset: consensus_data/1st_order/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Granite4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/consensus_data/1st_order/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

tidy_house_2nxc0tl14m
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260626-132136
  Model: smolvlm
  Dataset: consensus_data/1st_order/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma3...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/consensus_data/1st_order/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)..

...

crimson_kettle_djkkc0zvzw
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260626-132149
  Model: smolvlm
  Dataset: consensus_data/1st_order/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/consensus_data/1st_order/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

khaki_piano_xkgy8jrnqh
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260626-132202
  Model: smolvlm
  Dataset: consensus_data/1st_order/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/consensus_data/1st_order/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

strong_shark_mytmkjjlc2
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260626-132216
  Model: smolvlm
  Dataset: consensus_data/1st_order/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [20]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = Path(AML_OUTPUT + "/images")

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(str(TARGET_IMAGES_PATH))

for model_name, checkpoint, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )

EXTRACTION_DIRS = [f"../../outputs/extractions/{r}" for r in RUN_NAMES]

print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")


SmolVLM: 20260626-132115
Granite4: 20260626-132136
Gemma3: 20260626-132149
Gemma4: 20260626-132202
Ministral: 20260626-132216

Copy this block into the next cell:

RUN_NAMES = [
    "20260626-132115",
    "20260626-132136",
    "20260626-132149",
    "20260626-132202",
    "20260626-132216",
]



In [21]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = [
    "20260626-132115",
    "20260626-132136",
    "20260626-132149",
    "20260626-132202",
    "20260626-132216",
]

EXTRACTION_DIRS = [f"../../outputs/extractions/{r}" for r in RUN_NAMES]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

Using run names:
   20260626-132115
   20260626-132136
   20260626-132149
   20260626-132202
   20260626-132216


In [25]:
# When the jobs have completed successfully,
#  download the extractions so we can analyze them locally.
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260626-132115
         workers: 16


Finished[#############################################################]  100.0000%nstruct/20260626-132115/DRain_1951-1962_RainNos_971-1013_B015-10.json"[]  100.0000%%.0000%son"[]  100.0000%%000%"[]  100.0000%


[
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Cornwall-51.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Cumberland-109.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Cumberland-115.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Cumberland-239.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Devonshire_Part1-112.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Hampshire-142.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260626-132115/DRain_1871-1880_Kent_Part2-162.json",
  "Daily_rainfall_sample/outputs/extr

...

[
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Cornwall-51.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Cumberland-109.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Cumberland-115.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Cumberland-239.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Devonshire_Part1-112.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Hampshire-142.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_Kent_Part2-162.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260626-132202/DRain_1871-1880_London-153.json",
  "Daily_rainfall_sample

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260626-132216
         workers: 16


Finished[#############################################################]  100.0000%4B-Instruct-2503/20260626-132216/DRain_1951-1962_RainNos_971-1013_B015-10.json"[]  100.0000%%.0000%son"[]  100.0000%0000%"[]  100.0000%


[
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Cornwall-51.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Cumberland-109.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Cumberland-115.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Cumberland-239.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Devonshire_Part1-112.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1880_Hampshire-142.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260626-132216/DRain_1871-1

Build 1st-order consensus from downloaded extractions and check to see how well the models agree.

This produces outputs in `consensus_data/1st_order`.

In [27]:
cmd = [
    "bash",
    "../../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../../consensus_data",
    "--images-dir",
    "../../consensus_data/1st_order/images",
    "--variant-name",
    "1st_order",
    "--threshold",
    "3",
    "--null-threshold",
    "5",
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--validation-sample-denominator",
        "25",
    ]
)

subprocess.run(cmd, check=True)

=== Consensus Pipeline ===
Dataset root:      ../../consensus_data
Variant name:      1st_order
Threshold:         3
Null threshold:    5
Precision:         3
Extraction dirs:   5
Validate:          true
Validation sample: 1/25

Creating config...
✓ Created consensus config
  Variant:           1st_order
  Dataset root:      /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/consensus_data
  Variant dir:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/consensus_data/1st_order
  Config file:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/consensus_data/1st_order/consensus_config.json
  Extraction dirs:   5 model(s)
                     1. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260626-132115
                     2. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260626-132136
                     3. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/

CompletedProcess(args=['bash', '../../scripts/run_consensus_pipeline.sh', '--dataset-root', '../../consensus_data', '--images-dir', '../../consensus_data/1st_order/images', '--variant-name', '1st_order', '--threshold', '3', '--null-threshold', '5', '--precision', '3', '--extraction-dir', '../../outputs/extractions/20260626-132115', '--extraction-dir', '../../outputs/extractions/20260626-132136', '--extraction-dir', '../../outputs/extractions/20260626-132149', '--extraction-dir', '../../outputs/extractions/20260626-132202', '--extraction-dir', '../../outputs/extractions/20260626-132216', '--validate', '--validation-sample-denominator', '25'], returncode=0)

In [6]:
# Upload the sampled consensus transcriptions to the Azure ML datastore:
# Deleting the existing contents of the datastore directory first, if necessary.

import subprocess

subprocess.run(
    [
        "bash",
        "../../scripts/aml_delete.sh",
        AML_OUTPUT + "/transcriptions",
    ],
    check=True,
)
subprocess.run(
    [
        "bash",
        "../../scripts/aml_upload.sh",
        "--src",
        OUTPUT + "/consensus_transcriptions",
        "--dst",
        AML_OUTPUT + "/transcriptions",
    ],
    check=True,
)   


Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default
Deleting contents under datastore path: consensus_data/1st_order/transcriptions/
Done. Deleted contents under consensus_data/1st_order/transcriptions/
Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

Uploading  ../../consensus_data/1st_order/consensus_transcriptions
        → https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/transcriptions


Finished[#############################################################]  100.0000%_B015-10.json"[]  100.0000%%.0000%son"[]  100.0000%%000%"[]  100.0000%


[
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/transcriptions/DRain_1871-1880_Cornwall-51.json",
    "Last Modified": "2026-06-29T12:43:19+00:00",
    "Type": "application/json",
    "eTag": "\"0x8DED5DBFF669228\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/transcriptions/DRain_1871-1880_Cumberland-109.json",
    "Last Modified": "2026-06-29T12:43:19+00:00",
    "Type": "application/json",
    "eTag": "\"0x8DED5DBFF6E322D\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/transcriptions/DRain_1871-1880_Cumberland-115.json",
    "Last Modified": "2026-06-29T12:43:19+00:00",
    "Type": "application/json",
    "eTag": "\"0x8DED5DBFF7472D3\""
  },
  {
    "Blob": "https://sallmdatarescue02.blob.core.windows.net/default/consensus_data/1st_order/transcriptions/DRain_1871-1880_Cumberland-239.json",
    "Last Modified": "2026-0

CompletedProcess(args=['bash', '../../scripts/aml_upload.sh', '--src', '../../consensus_data/1st_order/consensus_transcriptions', '--dst', 'consensus_data/1st_order/transcriptions'], returncode=0)